# Google Maps Lane-Churn Detection — pipeline walkthrough

A short tour of the demo pipeline: synthetic GNSS traces → per-trace features → churn classifier → evaluation.

**Everything here is synthetic demo data.** The production results from Shashank's Google work (+20% recall / +14% accuracy) are stated in the README and are *not* the numbers this notebook produces.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.synth_traces import generate_segment_traces
from src.features.trace_features import FEATURE_COLUMNS, build_trace_features
from src.models.churn_classifier import build_models, train_models
from src.evaluate import evaluate_models, format_report
from sklearn.model_selection import train_test_split

## 1. Generate synthetic traces

200 road segments × 8 vehicle passes each. Each fix carries a lateral GNSS-to-map offset — the core churn signal — plus speed, heading, and its trace source (aGMM / iGMM).

In [ ]:
traces = generate_segment_traces(n_segments=200, seed=42)
print(f'{len(traces):,} fixes, {traces["trace_id"].nunique():,} traces')
traces.head()

## 2. Build per-trace features

One feature row per trace, computed from that trace alone — the per-trace streaming design. Offset statistics, offset drift along the segment, heading changes, speed, and source.

In [ ]:
features = build_trace_features(traces)
print(features.shape)
features.head()

## 3. Train the churn classifiers

Gradient boosting as the main model, logistic regression as the transparent baseline. Recall matters most here: a missed churn event leaves a stale lane on the map.

In [ ]:
X = features[FEATURE_COLUMNS]
y = features['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

models = train_models(build_models(), X_train, y_train)
print('trained:', list(models))

## 4. Evaluate

Demo metrics on synthetic data only — see the README for the separate production results from Google.

In [ ]:
results = evaluate_models(models, X_test, y_test)
print(format_report(results))